# Phase 9 : recommandations (chiffres de P9-14 et P9-15)

Ce notebook calcule les deux vérifications chiffrées de la phase 9 :

- **P9-15** : le score de risque désavantage-t-il un groupe de clients (genre, âge) ?
- **P9-14** : combien coûtent les fausses alertes selon le seuil, pour décider du seuil à recommander ?

Il part de la table d'analyse (P4-10) et des scores (P7-07). Le texte des recommandations est dans `docs/P9-02_recommandations.md` et `docs/P9-14_recommandations_operationnelles.md`.

## Chargement des données

In [1]:
import numpy as np
import pandas as pd

from src.load_data import load_clean_table
from src.comparisons import rate_table, chi2_test

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df = load_clean_table("analysis", columns=["id", "date", "gender", "age_at_transaction", "is_fraud"])
scores = load_clean_table("risk_scores", columns=["id", "flag_0_5", "flag_1", "flag_2"])
if not np.array_equal(df["id"].to_numpy(), scores["id"].to_numpy()):
    raise ValueError("Les scores ne sont pas dans le même ordre que la table d'analyse")
for col in ["flag_0_5", "flag_1", "flag_2"]:
    df[col] = scores[col].to_numpy()
del scores
df["groupe_age"] = pd.cut(df["age_at_transaction"], bins=[0, 29, 44, 59, 74, 120],
                          labels=["moins de 30 ans", "30 à 44 ans", "45 à 59 ans", "60 à 74 ans", "75 ans et plus"])
print(f"{len(df):,} transactions")

13,305,915 transactions


## P9-15 : usage éthique des variables dans le score

Décision : le genre et l'âge ne sont pas des variables du score de risque (P7-01 à P7-03) ni du modèle supervisé (P7-08). Il reste à vérifier que le score ne désavantage pas un groupe par un effet indirect.

Trois mesures au seuil de 1 % :

- le **taux de signalement** de chaque groupe ;
- le **taux de fausses alertes** : part des transactions étiquetées « pas de fraude » qui sont quand même signalées ;
- le **rappel** : part des fraudes signalées dans chaque groupe.

Le **rapport le plus bas / le plus haut** compare le groupe le moins signalé au plus signalé : plus il est proche de 1, plus le traitement est égal. La règle des quatre cinquièmes, utilisée aux États-Unis, considère qu'un rapport inférieur à 0,8 mérite une alerte.

Critère de fin : décision écrite et vérification chiffrée. Voir `docs/P9-15_ethique_variables.md`.

In [2]:
etiquetees = df[df["is_fraud"].notna()].assign(fraude=lambda t: t["is_fraud"].astype("int8"))
sans_fraude = etiquetees[etiquetees["fraude"] == 0]
fraudes = etiquetees[etiquetees["fraude"] == 1]

def equite(col):
    t = pd.DataFrame({
        "transactions": df.groupby(col, observed=True).size(),
        "taux de signalement (%)": 100 * df.groupby(col, observed=True)["flag_1"].mean(),
        "fausses alertes (%)": 100 * sans_fraude.groupby(col, observed=True)["flag_1"].mean(),
        "fraudes": fraudes.groupby(col, observed=True).size(),
        "rappel (%)": 100 * fraudes.groupby(col, observed=True)["flag_1"].mean(),
    })
    return t.round(3)

for col, titre in [("gender", "genre"), ("groupe_age", "groupe d'âge")]:
    t = equite(col)
    print(f"Par {titre} :")
    print(t.to_string())
    s = t["taux de signalement (%)"]
    print(f"Rapport le plus bas / le plus haut du taux de signalement : {s.min() / s.max():.2f}")
    print()

Par genre :
        transactions  taux de signalement (%)  fausses alertes (%)  fraudes  rappel (%)
gender                                                                                 
Female       6815916                    1.016                0.987     6982      22.959
Male         6489999                    0.983                0.951     6350      25.008
Rapport le plus bas / le plus haut du taux de signalement : 0.97

Par groupe d'âge :
                 transactions  taux de signalement (%)  fausses alertes (%)  fraudes  rappel (%)
groupe_age                                                                                      
moins de 30 ans       1323856                    1.108                1.085     1052      24.905
30 à 44 ans           4545633                    1.056                1.029     4140      24.541
45 à 59 ans           4296742                    0.979                0.945     4500      23.600
60 à 74 ans           1991021                    0.979            

In [3]:
pd.DataFrame([{**chi2_test(df, col, "flag_1"), "variable": nom}
              for col, nom in [("gender", "genre"), ("groupe_age", "groupe d'âge")]])

,variable,test,statistique,ddl,p_valeur,taille d'effet,mesure,lecture
0,genre,khi-deux,36.3,1,1.713438e-09,0.002,V de Cramér,négligeable
1,groupe d'âge,khi-deux,932.5,4,1.506493e-200,0.008,V de Cramér,négligeable


**Constats**

- **Genre.** Les femmes et les hommes sont signalés presque autant : 1,016 % et 0,983 % (rapport de 0,97). Les fausses alertes (0,987 % et 0,951 %) et le rappel (22,959 % et 25,008 %) sont proches. Aucun groupe n'est désavantagé.
- **Âge.** Le taux de signalement baisse avec l'âge : 1,108 % avant 30 ans, 0,771 % à 75 ans et plus. Le rapport le plus bas / le plus haut vaut 0,70, sous le seuil d'alerte de 0,8.
- Les clients de moins de 30 ans reçoivent donc plus de fausses alertes (1,085 % contre 0,729 % à 75 ans et plus), alors que l'âge n'est pas dans le score. Le rappel, lui, ne dépend pas de l'âge (de 22,198 % à 25,369 %).
- Les tailles d'effet restent négligeables (V de Cramér de 0,002 pour le genre et 0,008 pour l'âge), mais l'écart entre les âges doit être suivi.

## P9-14 : coût des fausses alertes selon le seuil

Pour chaque seuil, à partir des transactions étiquetées : fraudes trouvées, fraudes ratées et fausses alertes. Le nombre d'alertes par jour vient de toutes les transactions (P7-04).

Le temps de contrôle d'une alerte n'est pas connu : **hypothèse de 5 minutes par alerte**, à remplacer par la mesure réelle de l'équipe de la banque.

Critère de fin de P9-14 : tableau des recommandations opérationnelles validé. Voir `docs/P9-14_recommandations_operationnelles.md`.

In [4]:
MINUTES_PAR_ALERTE = 5  # hypothèse, à remplacer par la mesure réelle
jours = df["date"].dt.normalize().nunique()
lignes = []
for seuil, col in [("0,5 %", "flag_0_5"), ("1 %", "flag_1"), ("2 %", "flag_2")]:
    alertes_jour = df[col].sum() / jours
    sig = etiquetees[etiquetees[col] == 1]
    precision = sig["fraude"].mean()
    trouvees = int(sig["fraude"].sum())
    lignes.append({
        "seuil": seuil,
        "alertes par jour": round(alertes_jour, 1),
        "fausses alertes par jour": round(alertes_jour * (1 - precision), 1),
        "heures de contrôle par jour": round(alertes_jour * MINUTES_PAR_ALERTE / 60, 1),
        "fraudes trouvées (étiquetées)": trouvees,
        "fraudes ratées (étiquetées)": int(fraudes.shape[0] - trouvees),
        "fausses alertes pour une fraude trouvée": round((1 - precision) / precision, 1),
    })
cout = pd.DataFrame(lignes).set_index("seuil")
print(f"Jours couverts : {jours:,} ; fraudes étiquetées : {fraudes.shape[0]:,}")
cout

Jours couverts : 3,591 ; fraudes étiquetées : 13,332


,alertes par jour,fausses alertes par jour,heures de contrôle par jour,fraudes trouvées (étiquetées),fraudes ratées (étiquetées),fausses alertes pour une fraude trouvée
seuil,,,,,,
"0,5 %",18.5,17.5,1.5,2608,10724,16.1
1 %,37.1,35.7,3.1,3191,10141,27.0
2 %,74.1,71.9,6.2,5361,7971,32.3


**Constats**

- Au seuil de 1 %, l'équipe reçoit 37,1 alertes par jour, dont 35,7 fausses alertes. Avec l'hypothèse de 5 minutes par alerte, le contrôle demande 3,1 heures par jour.
- Il faut contrôler 27,0 fausses alertes pour trouver une fraude. Sur les transactions étiquetées, le seuil de 1 % trouve 3 191 fraudes et en rate 10 141.
- Au seuil de 0,5 %, la charge est divisée par deux (18,5 alertes et 1,5 heure par jour) et chaque alerte est plus utile (16,1 fausses alertes pour une fraude), mais 2 608 fraudes seulement sont trouvées.
- Au seuil de 2 %, on trouve 5 361 fraudes, mais il faut 74,1 alertes et 6,2 heures de contrôle par jour (32,3 fausses alertes pour une fraude).